# Proyecto 1 — LSA con SVD

## 1. Elección y descripción del corpus

Elijan un corpus con al menos 50
documentos. Cada documento debe tener una cantidad razonable de texto y la
fuente debe ser abierta.
Describan la fuente de los datos, indiquen cuantos documentos usaran y
expliquen por que el corpus es interesante. Incluyan una tabla o resumen con el
numero de documentos y la fuente.
Algunos ejemplos posibles son abstracts de articulos cientificos, noticias, dis-
cursos politicos, letras de canciones, reviews de productos u otro corpus publico
que el grupo considere apropiado. Pueden buscar datos en Kaggle, Hugging Face
Datasets u otras fuentes abiertas.

In [1]:
import pandas as pd
import os
import re

In [3]:
df = pd.read_csv(os.path.join("data", "song_lyrics.csv"), nrows=1000000) 
df = df[df["language"] == "es"]
df = df[df["language_cld3"] == "es"]
df = df[df["language_ft"] == "es"]
print(len(df))
#IMPORTANTE: si uno no pone limite, simplemente ningun computador es capaz de procesar tantas canciones, por ende solo analizaremos 1 M
df.head()

35333


,title,tag,artist,year,views,features,lyrics,id,language_cld3,language_ft,language
4084,Gasolina Remix,rap,Daddy Yankee,2004,47419,"{""Lil Jon"",N.O.R.E.,Pitbull}","Letra de ""Gasolina (Remix)"" ft. N.O.R.E., Lil ...",4171,es,es,es
4088,Rompe,rap,Daddy Yankee,2005,18308,{},"[Letra de ""Rompe""]\n\n[Intro]\nYou know\nLos c...",4175,es,es,es
4101,We No Speak Americano Remix,rap,Pitbull,2011,3515,{},[Intro - Pitbull]\nThis is worldwide\nEso aqui...,4190,es,es,es
5395,Suenos en Realidad,rap,Ozomatli,2011,68,{},Todos que convierten suenos en realidad\nSu lu...,36381,es,es,es
11985,Triste Me Pongo,rap,Califas (Mr. Lil One & Shisty),2011,181,{},{Girl Singing Chorus 2X}\nEstoy tan enamorada ...,12570,es,es,es


In [4]:
letras = [letra for letra in df["lyrics"]]
letras[0]

'Letra de "Gasolina (Remix)" ft. N.O.R.E., Lil Jon & Pitbull\n\n[Intro: Lil Jon, Pitbull, Daddy Yankee]\nYeah!\nRemix!\nRemix!\nRemix!\n(Who\'s this?)\nWhat\'s his name?\n(Da-ddy Yan-kee!)\n\nDaddy Yankee\nBoricua, Cubano, Boricua, Cubano, Boricua, Cubano, Boricua\n\n[Pre-Verso: Daddy Yankee]\nZúmbale mambo pa\' que mis gatas prendan los motores\nZúmbale mambo pa\' que mis gatas prendan los motores\nZúmbale mambo pa\' que mis gatas prendan los motores\nQue se preparen que lo que viene es pa\' que le den\n\n[Verso 1: Daddy Yankee, Lil Jon]\n(¡Duro!) Mamita, yo sé que tú no te me va\' a quitar\n(¡Duro!) Lo que me gusta es que tú te dejas llevar\n(¡Duro!) Todos los weekend\'es ella sale a vacilar\n(¡Duro!) Mi gata no para de janguear, porque (Yeah!)\n[Coro: Daddy Yankee, Glory, (Lil Jon)]\nA ella le gusta la gasolina (whatcha say?)\nDame más gasolina (hey)\nA ella encanta la gasolina (whatcha say?)\nDame más gasolina (hey)\n\nA ella le gusta la gasolina (yeah!)\nDame más gasolina (yeah!)\

Se usaran las letras de 35333 canciones, poemas y libros (en su mayoria canciones) en español (filtradas manualmente en la celda anterior) de mas de 5 millones de datos recopilados por Genius en un dataset de Kaggle, del autor Nikhil Nayak, https://www.kaggle.com/datasets/carlosgdcj/genius-song-lyrics-with-language-information?resource=download . Es interesante ya que junta creaciones artisticas de todos los generos y estilos de todo el mundo, con gran variedad de años y artistas, de las canciones mas famosas a las desconocidas.

carlosgdcj. (2023). Genius song lyrics with language information. Kaggle. https://www.kaggle.com/datasets/carlosgdcj/genius-song-lyrics-with-language-information

In [5]:
# Como el archivo original es muy pesado para subirlo al repositorio se creara una copia filtrada solo 
# con las letras que se usara en el proyecto para tener un archivo en el repositorio y se pueda replicar.
df.to_csv(os.path.join("data", "songs_lyrics_filtred.csv"))

## 2. Hipotesis inicial

Antes de realizar el analisis, propongan una hipotesis
sobre la estructura del corpus.

Por ejemplo, pueden anticipar temas posibles, grupos de documentos, palabras importantes o separaciones esperadas entre categorias. La salida esperada
es un parrafo corto con una hipotesis inicial concreta

Como hipotesis inicial buscaremos predecir la epoca del texto, (50s, 80s, 20s, etc), ya que el corpus contiene textos literarios de varias epocas, donde probablemente tengan una forma de hablar o un genero habitual distinto entre epoca y epoca.


## 3. Preprocesamiento del texto

Limpien el texto para construir un vocabulario util.

Como minimo, expliquen que decisiones tomaron sobre tokenizacion, normalizacion, eliminacion de stopwords y eliminacion de terminos poco informativos.

Se recomienda usar TF-IDF en lugar de conteos crudos. Tambien pueden incluirlematizacion y n-gramas si esto ayuda al corpus elegido.

Reporten el tamano del vocabulario final y expliquen brevemente por que el preprocesamiento elegido es razonable

In [8]:
# Primero para tokenizar separamos por palabras, dejando palabras como remix, ft, intro, etc, ya que igual puede ser una diferencia de epoca el usarlo
# y sacando los caracteres no alfanumericos como : o . , ya que no aportan informacion util

letras_filtradas = list()

for texto in letras:
    nuevo = list()
    # Usamos re para separar las palabras con varios posibles caracteres
    texto_separado = re.split(r"[ ,\n.:]", texto)
    for palabra in texto_separado:
        palabra_alfanumerica = str()
        for letra in palabra:
            if letra.isalpha() or letra.isdigit():
                palabra_alfanumerica += letra
        if len(palabra_alfanumerica) > 1:
            # solo agrega la palabra si tiene contenido util y no esta vacio o con solo 1 letra
            nuevo.append(palabra_alfanumerica)
    letras_filtradas.append(nuevo)

letras_filtradas[0]

['Letra',
 'de',
 'Gasolina',
 'Remix',
 'ft',
 'Lil',
 'Jon',
 'Pitbull',
 'Intro',
 'Lil',
 'Jon',
 'Pitbull',
 'Daddy',
 'Yankee',
 'Yeah',
 'Remix',
 'Remix',
 'Remix',
 'Whos',
 'this',
 'Whats',
 'his',
 'name',
 'Daddy',
 'Yankee',
 'Daddy',
 'Yankee',
 'Boricua',
 'Cubano',
 'Boricua',
 'Cubano',
 'Boricua',
 'Cubano',
 'Boricua',
 'PreVerso',
 'Daddy',
 'Yankee',
 'Zúmbale',
 'mambo',
 'pa',
 'que',
 'mis',
 'gatas',
 'prendan',
 'los',
 'motores',
 'Zúmbale',
 'mambo',
 'pa',
 'que',
 'mis',
 'gatas',
 'prendan',
 'los',
 'motores',
 'Zúmbale',
 'mambo',
 'pa',
 'que',
 'mis',
 'gatas',
 'prendan',
 'los',
 'motores',
 'Que',
 'se',
 'preparen',
 'que',
 'lo',
 'que',
 'viene',
 'es',
 'pa',
 'que',
 'le',
 'den',
 'Verso',
 'Daddy',
 'Yankee',
 'Lil',
 'Jon',
 'Duro',
 'Mamita',
 'yo',
 'sé',
 'que',
 'tú',
 'no',
 'te',
 'me',
 'va',
 'quitar',
 'Duro',
 'Lo',
 'que',
 'me',
 'gusta',
 'es',
 'que',
 'tú',
 'te',
 'dejas',
 'llevar',
 'Duro',
 'Todos',
 'los',
 'weekendes',

In [9]:
# Normalizamos las palabras haciendolas todas con letras minusculas
for texto in range(len(letras_filtradas)):
    for palabra in range(len(letras_filtradas[texto])):
        letras_filtradas[texto][palabra] = letras_filtradas[texto][palabra].lower()

letras_filtradas[0]

['letra',
 'de',
 'gasolina',
 'remix',
 'ft',
 'lil',
 'jon',
 'pitbull',
 'intro',
 'lil',
 'jon',
 'pitbull',
 'daddy',
 'yankee',
 'yeah',
 'remix',
 'remix',
 'remix',
 'whos',
 'this',
 'whats',
 'his',
 'name',
 'daddy',
 'yankee',
 'daddy',
 'yankee',
 'boricua',
 'cubano',
 'boricua',
 'cubano',
 'boricua',
 'cubano',
 'boricua',
 'preverso',
 'daddy',
 'yankee',
 'zúmbale',
 'mambo',
 'pa',
 'que',
 'mis',
 'gatas',
 'prendan',
 'los',
 'motores',
 'zúmbale',
 'mambo',
 'pa',
 'que',
 'mis',
 'gatas',
 'prendan',
 'los',
 'motores',
 'zúmbale',
 'mambo',
 'pa',
 'que',
 'mis',
 'gatas',
 'prendan',
 'los',
 'motores',
 'que',
 'se',
 'preparen',
 'que',
 'lo',
 'que',
 'viene',
 'es',
 'pa',
 'que',
 'le',
 'den',
 'verso',
 'daddy',
 'yankee',
 'lil',
 'jon',
 'duro',
 'mamita',
 'yo',
 'sé',
 'que',
 'tú',
 'no',
 'te',
 'me',
 'va',
 'quitar',
 'duro',
 'lo',
 'que',
 'me',
 'gusta',
 'es',
 'que',
 'tú',
 'te',
 'dejas',
 'llevar',
 'duro',
 'todos',
 'los',
 'weekendes',

## 4. Construccion de la matriz textual

Construyan la matriz que sera analizada con SVD.

Definan claramente si trabajaran con una matriz termino-documento o documento-palabra. En la clase usamos la convencion

$$A \in \mathbb{R}^{m \times n},$$

donde las filas representan terminos, las columnas representan documentos, $m$ es el numero de terminos y $n$ es el numero de documentos. Si usan scikit-learn, recuerden que sus vectorizadores suelen entregar una matriz con documentos en las filas y terminos en las columnas; pueden transponerla si quieren seguir la convencion del curso.

Reporten la dimension de la matriz y expliquen como fue construida. Si usan TF-IDF, definan brevemente que representa una entrada $A_{ij}$.

## 5. Calculo de la SVD

Calculen la descomposicion

$$A = U\Sigma V^T.$$

Expliquen que representan los valores singulares en este problema y que informacion entregan los vectores singulares izquierdos y derechos. Si $A$ es una matriz termino-documento, las columnas de $U$ describen direcciones principales en el espacio de terminos, mientras que las columnas de $V$ describen direcciones principales en el espacio de documentos.

## 6. Valores singulares y eleccion de dimension

Analicen cuanto de la estructura del problema se concentra en pocas componentes.
Grafiquen los valores singulares, observen su decaimiento y elijan un valor razonable de $k$ para una representacion reducida. Justifiquen brevemente la eleccion de $k$.

## 7. Representacion de documentos y terminos en baja dimension

Usen componentes de la SVD para construir representaciones reducidas.
Para visualizar en dos dimensiones, pueden elegir cualquier par de componentes. En particular, es natural comenzar con las dos primeras componentes, porque corresponden a los dos valores singulares mas grandes. Si siguen la convencion termino-documento de la clase y escriben

$$A_2 = U_2 \Sigma_2 V_2^T,$$

para las dos primeras componentes, entonces las coordenadas de los documentos estan dadas por

$$D_2 = \Sigma_2 V_2^T \in \mathbb{R}^{2 \times n},$$

y las coordenadas de los terminos estan dadas por

$$T_2 = U_2 \Sigma_2 \in \mathbb{R}^{m \times 2}.$$

Mas generalmente, si quieren visualizar las componentes $a$ y $b$, usen las columnas correspondientes de $U$, las filas correspondientes de $V^T$ y los valores singulares $\sigma_a, \sigma_b$.
Incluyan una visualizacion 2D de documentos y una visualizacion 2D de palabras o terminos relevantes, indicando que par de componentes usaron. Interpreten agrupamientos, separaciones o asociaciones observadas.

## 8. Exploracion semantica

Usen la representacion reducida para explorar relaciones en el corpus.
Incluyan 2 o 3 ejemplos concretos, como palabras cercanas, documentos cercanos, componentes con posible significado tematico o grupos de documentos que parezcan relacionados. Expliquen por que esos ejemplos son coherentes o sorprendentes.

## 9. Discusion, limitaciones y conclusiones

Comparen los resultados con la hipotesis inicial.
Indiquen si la hipotesis se cumple o no, expliquen resultados esperados e inesperados, y discutan limites del metodo y del corpus. Recuerden que LSA no reemplaza la lectura del corpus: sirve como herramienta exploratoria para descubrir patrones, formular hipotesis y construir visualizaciones de baja dimension.